In this notebook I will load the neural data from the EMU cohort, and make figures for the paper:
    

In [ ]:
%matplotlib inline
%reload_ext autoreload
%autoreload 2

In [ ]:
import numpy as np
import mne
import mne_connectivity
from mne_connectivity import phase_slope_index, seed_target_indices, spectral_connectivity_epochs
from glob import glob
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
import seaborn as sns
from scipy.stats import zscore, linregress, ttest_ind, ttest_rel, ttest_1samp
import pandas as pd
from mne.preprocessing.bads import _find_outliers
import fooof
# Import plotting functions
from fooof.plts.spectra import plot_spectrum, plot_spectra
# Import the FOOOF object
from fooof import FOOOF
from fooof import FOOOFGroup
import os 
import joblib
import statsmodels.formula.api as smf
import statsmodels.api as sm
import pickle
from tqdm import tqdm
from IPython.display import clear_output
import warnings 

# I only want to see warnings once
warnings.filterwarnings('ignore')

from joblib import delayed, Parallel
from statsmodels.stats import multitest


In [ ]:
import sys
sys.path.append('/Users/salmanqasim/Documents/GitRepos/LFPAnalysis')

In [ ]:
from LFPAnalysis import lfp_preprocess_utils, sync_utils, analysis_utils, nlx_utils, oscillation_utils, statistics_utils


In [ ]:
# Set some plot defaults for matplotlib (when not using seaborn): 

import  matplotlib.font_manager

# path = '/sc/arion/projects/guLab/Salman/Arial.ttf'
# matplotlib.font_manager.fontManager.addfont(path)

# prop = matplotlib.font_manager.FontProperties(fname=path)

# # Set the default font family to the registered custom font
# plt.rcParams['font.family'] = prop.get_name()

# Rest of your customization code
plt.rcParams['font.size'] = 20
plt.rcParams['lines.linewidth'] = 1

# Set palette: 

# Define RGB values for your two colors
ignacio_purple = np.array([86, 28, 70]) / 255
ignacio_blue = np.array([0, 70, 126]) / 255 

# Create a Seaborn palette with your custom colors normalized to 0-1
custom_palette = sns.color_palette([ignacio_purple, ignacio_blue])

ignacio_orange = np.array([255, 172, 77]) / 255
ignacio_mauve = np.array([193, 155, 211]) / 255
ignacio_beige = np.array([205, 182, 162]) / 255

# Set the custom palette
sns.set_palette(custom_palette)
sns.set_context("paper")

# from YBA
hpc_c = np.array([0.44314, 0.2902, 1])
amy_c = np.array([0.49412, 0, 0.90196])
ofc_c = np.array([1, 0.75686, 0.41569])
acc_c = np.array([0.7098, 0.27059 ,0.68235])
dmpfc_c = np.array([0.40784, 0.01961, 0])
dlpfc_c = np.array([1, 0.06275, 0.06275])


In [ ]:
# plot the brain region swatches for figure design

In [ ]:
# Create a color palette
palette = sns.color_palette([dmpfc_c, dlpfc_c, ofc_c, acc_c, amy_c, hpc_c])

# Display the color swatches
sns.palplot(palette)
plt.show()

In [ ]:
base_dir = '/Users/salmanqasim/Downloads/mybox-selected' # this is the root directory for most un-archived data and results


In [ ]:
# try to load the neural data for Day 1 
import neo

# datapath = f'{base_dir}/MemoryBanditDay1/ECOG001/ECOGS001R06.dat'
datapath = f'{base_dir}/MemoryBanditDay2/ECOG001/ECOGS001R01.dat'

# # use bci2000 loading format
# reader = neo.io.BCI2000IO(datapath)
# segment = reader.read_segment(lazy=True)

# ch_names =reader.header['signal_channels']['name']
# fs = reader.header['signal_channels']['sampling_rate']

# signal = segment.analogsignals[0]  # neo.core.AnalogSignal object
# signals = signal.load()



In [ ]:
# use bci2000 loading format
reader = neo.io.BCI2000IO(datapath)
segment = reader.read_segment(lazy=True)

ch_names =reader.header['signal_channels']['name']

In [ ]:
from BCI2kReader import BCI2kReader as b2k

with b2k.BCI2kReader(datapath) as f:
    my_signals, my_states=f.readall() #
    # ch_names = f.name


In [ ]:
np.where(ch_names=='LAH1')[0]

In [ ]:
plt.plot(my_states['StimulusCode'][0, :])

In [ ]:
# I need to sync the neural data to the behavioral data 
# laod the behavioral data 
behavioral_df = pd.read_csv(f'{base_dir}/MemoryBanditDay2/UAB018_MEM2_2025-11-18_11h00.10.065.csv')

In [ ]:
behavioral_df.keys()

In [ ]:
behavioral_df

# TTL_1 = behavioral_df.filter(like='sync1.started').dropna().values
# TTL_2 = behavioral_df.filter(like='sync2.started').dropna().values
# TTL_3 = behavioral_df.filter(like='sync3.started').dropna().values
# TTL_4 = behavioral_df.filter(like='sync4.started').dropna().values
# TTL_5 = behavioral_df.filter(like='sync5.started').dropna().values

beh_ts = np.sort(np.squeeze(TTL_3).tolist() + np.squeeze(TTL_4).tolist())  



In [ ]:
# get the first rise for each event, leaving out the adjacent entries that have diff==1
signal = my_states['StimulusCode'][0, :]
# Find transitions from 0 (or not-1) to 1 - these are the leading edges
leading_edges = np.where(np.diff(signal) == 1)[0] + 1
# Handle case where signal starts at 1
if signal[0] == 1:
    leading_edges = np.concatenate([[0], leading_edges])

print(f"Found {len(leading_edges)} leading edges")
print(f"First few leading edges: {leading_edges[:10]}")

# Plot signal with leading edges marked
plt.figure(figsize=(12, 4))
plt.plot(my_states['StimulusCode'][0, :], label='Signal', alpha=0.7)
plt.scatter(leading_edges, np.ones(len(leading_edges)), 
            color='red', marker='v', s=100, label='Leading Edges', zorder=5)
plt.xlabel('Sample')
plt.ylabel('Signal Value')
plt.title('Trigger Signal with Leading Edges')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:
sns.regplot(x=beh_ts, y=leading_edges[1:-2])

In [ ]:
bfix = beh_ts[0]
res = linregress(beh_ts-bfix, leading_edges[1:-2])
slope = res[0]
offset = res[1]
offset = offset - bfix*slope
rval = res[2]

In [ ]:
image_presentation = [(x*slope + offset) for x in behavioral_df['image.started'].dropna().values]


In [ ]:
image_times = np.array(image_presentation)

In [ ]:
# Create MNE epochs from image_times for each channel
fs = 2048  # sampling frequency

# Check the shape of my_signals to ensure correct orientation
# MNE expects (n_channels, n_times) format
if my_signals.shape[0] > my_signals.shape[1]:
    # If first dimension is larger, likely (n_times, n_channels), need to transpose
    signals_data = my_signals.T
else:
    # Already in (n_channels, n_times) format
    signals_data = my_signals

# Ensure we have the right number of channels
n_channels = len(ch_names)
if signals_data.shape[0] != n_channels:
    # If mismatch, try transposing
    signals_data = signals_data.T

# Create MNE Info object
info = mne.create_info(ch_names=ch_names.tolist(), sfreq=fs, ch_types='ecog')

# Create MNE Raw object
raw = mne.io.RawArray(signals_data, info)

# Create events from image_times (convert sample indices to events)
# image_times are already in sample units
events = np.zeros((len(image_times), 3), dtype=int)
events[:, 0] = image_times.astype(int)  # event time in samples
events[:, 2] = 1  # event id (all set to 1 for image presentation)

# Define epoch time window (e.g., -0.5 to 2.0 seconds around image presentation)
tmin = -0.5  # start 0.5 seconds before
tmax = 2.0   # end 2.0 seconds after

# Create epochs
epochs = mne.Epochs(raw, events, event_id={'image': 1}, tmin=tmin, tmax=tmax, 
                    baseline=None, preload=True, proj=False)

print(f"Created epochs with shape: {epochs.get_data().shape}")
print(f"Number of epochs: {len(epochs)}")
print(f"Number of channels: {len(epochs.ch_names)}")
print(f"Time range: {epochs.tmin} to {epochs.tmax} seconds")


In [ ]:
# 1. Apply notch filter for 60 Hz noise
print("Applying notch filter at 60 Hz...")
# Use mne.filter.notch_filter on the epochs data
epochs_notched = epochs.copy()
epochs_data = epochs_notched.get_data()  # shape: (n_epochs, n_channels, n_times)
# Reshape to 2D for filtering: (n_epochs * n_channels, n_times)
n_epochs, n_channels, n_times = epochs_data.shape
data_2d = epochs_data.reshape(n_epochs * n_channels, n_times)
# Apply notch filter (Fp is sampling frequency, first positional arg after data)
filtered_data_2d = mne.filter.notch_filter(data_2d, fs, freqs=[60, 120, 180], 
                                           notch_widths=2, verbose=False)
# Reshape back to 3D
filtered_data = filtered_data_2d.reshape(n_epochs, n_channels, n_times)
epochs_notched._data = filtered_data
print("Notch filtering complete.")

# Downsample to 512 Hz
print("Downsampling to 512 Hz...")
target_sfreq = 512
epochs_notched = epochs_notched.resample(target_sfreq, npad='auto', verbose=False)
fs = target_sfreq  # Update sampling frequency
print(f"Downsampling complete. New sampling frequency: {fs} Hz")

# 2. Compute HFA (High Frequency Activity) for each epoch
# HFA is typically computed as power in the 70-200 Hz range
print("\nComputing HFA...")
hfa_fmin = 70  # lower bound for HFA
hfa_fmax = 150  # upper bound for HFA

# Filter to HFA band and compute envelope (Hilbert transform)
epochs_hfa = epochs_notched.copy().filter(l_freq=hfa_fmin, h_freq=hfa_fmax, 
                                           method='iir', verbose=False)

# Compute HFA power using Hilbert transform (envelope)
from scipy.signal import hilbert

hfa_data = epochs_hfa.get_data()  # shape: (n_epochs, n_channels, n_times)
hfa_power = np.abs(hilbert(hfa_data, axis=-1)) ** 2  # compute power envelope

# Smooth HFA using moving mean filter with 200 ms window and 150 ms overlap
print("Smoothing HFA with moving mean filter...")
from scipy.ndimage import uniform_filter1d

window_size_ms = 200  # 200 ms window
overlap_ms = 150  # 150 ms overlap

# Convert to samples
window_size_samples = int(window_size_ms * fs / 1000)

# Apply moving mean filter using uniform_filter1d (equivalent to convolution with boxcar)
# This creates a moving average where each point is the mean of the window centered at that point
# The windows naturally overlap
hfa_power = uniform_filter1d(hfa_power, size=window_size_samples, axis=-1, mode='constant')
print(f"Smoothing complete. Window size: {window_size_ms} ms ({window_size_samples} samples)")

# 3. Baseline correct to [-0.5, 0] period
print("Baseline correcting HFA to [-0.5, 0] period...")
times = epochs_hfa.times
baseline_mask = (times >= -0.5) & (times <= 0)

# Compute baseline mean for each epoch and channel
baseline_mean = np.nanmean(hfa_power[:, :, baseline_mask], axis=2, keepdims=True)

# Baseline correct (subtract baseline mean, then divide by it for percent change)
# Using percent change: (signal - baseline) / baseline * 100
hfa_baselined = ((hfa_power - baseline_mean) / baseline_mean) * 100

print(f"HFA computed. Shape: {hfa_baselined.shape}")
print(f"Time range: {times[0]:.3f} to {times[-1]:.3f} seconds")


In [ ]:
hfa_baselined.shape

In [ ]:
# Fix: Properly split HFA data by response type
# hfa_baselined shape: (80, n_channels, n_times) - one for each of the 80 trials
# Result: hfa_by_response['left'] shape: (n_left, n_channels, n_times)
#         hfa_by_response['right'] shape: (n_right, n_channels, n_times)

# Ensure we have exactly 80 responses matching the 80 epochs
assert len(responses) == hfa_baselined.shape[0], f"Mismatch: {len(responses)} responses but {hfa_baselined.shape[0]} epochs"

# Re-split HFA data by response type correctly
hfa_by_response = {}
for resp in unique_responses:
    resp_mask = responses == resp
    n_trials = resp_mask.sum()
    if n_trials > 0:
        hfa_by_response[resp] = hfa_baselined[resp_mask, :, :]  # shape: (n_trials, n_channels, n_times)
        print(f"  Response '{resp}': {n_trials} trials, shape {hfa_by_response[resp].shape}")

# Recompute ERP for each response type (average across trials)
hfa_erp_by_response = {resp: np.mean(data, axis=0) for resp, data in hfa_by_response.items()}


In [ ]:
# Create individual plots for each channel and save as PDFs
import os

n_channels_to_plot = 196
n_channels = min(n_channels_to_plot, hfa_baselined.shape[1])
save_dir = "/Users/salmanqasim/Downloads/mybox-selected/MemoryBanditDay2"

# Create directory if it doesn't exist
os.makedirs(save_dir, exist_ok=True)

print(f"Creating individual plots for {n_channels} channels...")
for ch_idx in range(n_channels):
    ch_name = epochs_hfa.ch_names[ch_idx]
    
    # Create figure for this channel
    fig, ax = plt.subplots(1, 1, figsize=(8, 6))
    
    # Plot left responses
    if 'left' in hfa_by_response:
        left_data = hfa_by_response['left'][:, ch_idx, :]  # shape: (n_left_trials, n_times)
        left_mean = np.mean(left_data, axis=0)
        left_sem = np.std(left_data, axis=0) / np.sqrt(left_data.shape[0])
        ax.plot(times, left_mean, color=response_colors['left'], 
               label=f"Left (n={left_data.shape[0]})", linewidth=2)
        ax.fill_between(times, left_mean - left_sem, left_mean + left_sem, 
                       color=response_colors['left'], alpha=0.3)
    
    # Plot right responses
    if 'right' in hfa_by_response:
        right_data = hfa_by_response['right'][:, ch_idx, :]  # shape: (n_right_trials, n_times)
        right_mean = np.mean(right_data, axis=0)
        right_sem = np.std(right_data, axis=0) / np.sqrt(right_data.shape[0])
        ax.plot(times, right_mean, color=response_colors['right'], 
               label=f"Right (n={right_data.shape[0]})", linewidth=2)
        ax.fill_between(times, right_mean - right_sem, right_mean + right_sem, 
                       color=response_colors['right'], alpha=0.3)
    
    # Add reference lines
    ax.axhline(y=0, color='k', linestyle='--', linewidth=1, alpha=0.5)
    ax.axvline(x=0, color='r', linestyle='--', linewidth=1, alpha=0.5, label='Image onset')
    ax.axvspan(-0.5, 0, alpha=0.1, color='gray', label='Baseline period')
    
    # Formatting
    ax.set_xlabel('Time (s)', fontsize=12)
    ax.set_ylabel('HFA (% change from baseline)', fontsize=12)
    ax.set_title(f'{ch_name}: HFA Left vs Right Responses', fontsize=14)
    ax.legend(fontsize=10, loc='best')
    ax.grid(True, alpha=0.3)
    sns.despine()
    
    # Save as PDF
    pdf_path = os.path.join(save_dir, f'{ch_name}_HFA_left_right.pdf')
    plt.tight_layout()
    plt.savefig(pdf_path, dpi=300, format='pdf', bbox_inches='tight')
    plt.close()  # Close figure to free memory
    
    if (ch_idx + 1) % 20 == 0:
        print(f"  Saved {ch_idx + 1}/{n_channels} channels...")

print(f"\nAll plots saved to: {save_dir}")


In [ ]:
hfa_by_response['left'].shape


In [ ]:
# Get behavioral responses corresponding to each epoch
# Epochs were created from behavioral_df['image.started'].dropna(), so we need to match them
image_started_indices = behavioral_df['image.started'].dropna().index
responses = behavioral_df.loc[image_started_indices, 'key_resp.keys'].values

# Get unique responses and create color mapping
unique_responses = pd.Series(responses).dropna().unique()
n_responses = len(unique_responses)
print(f"Found {n_responses} unique response types: {unique_responses}")

# Create color palette for responses
if n_responses <= len(custom_palette):
    response_colors = dict(zip(unique_responses, custom_palette[:n_responses]))
else:
    # Use seaborn palette if more responses than custom colors
    response_colors = dict(zip(unique_responses, sns.color_palette("husl", n_responses)))

# Split HFA data by response type
hfa_by_response = {}
for resp in unique_responses:
    resp_mask = responses == resp
    if resp_mask.sum() > 0:
        hfa_by_response = hfa_baselined[resp_mask, :, :]  # shape: (n_epochs, n_channels, n_times)

# Compute ERP for each response type
hfa_erp_by_response = {resp: np.mean(data, axis=0) for resp, data in hfa_by_response.items()}


In [ ]:
hfa_erp_by_response['left'].shape

In [ ]:
plt.plot(hfa_erp_by_response['left'][0])
plt.plot(hfa_erp_by_response['right'][0])

In [ ]:
# Re-plot with response splitting
# Update the channel plots to show different responses with different colors
# Only plot the first 196 channels
n_channels_to_plot = 196
n_channels = min(n_channels_to_plot, hfa_baselined.shape[1])
n_cols = 10
n_rows = int(np.ceil(n_channels / n_cols))

fig, axes = plt.subplots(n_rows, n_cols, figsize=(20, 2*n_rows), sharex=True, sharey=True)
axes = axes.flatten() if n_channels > 1 else [axes]

# Plot each channel with different colors for different responses
for ch_idx, ch_name in enumerate(epochs_hfa.ch_names[:n_channels]):
    ax = axes[ch_idx]
    for resp in unique_responses:
        if resp in hfa_erp_by_response:
            hfa_erp_resp = hfa_erp_by_response[resp]
            ax.plot(times, hfa_erp_resp[ch_idx, :], linewidth=1.5, 
                   color=response_colors[resp], label=str(resp))
    ax.axhline(y=0, color='k', linestyle='--', linewidth=0.5, alpha=0.5)
    ax.axvline(x=0, color='r', linestyle='--', linewidth=0.5, alpha=0.5)  # mark image onset
    ax.set_title(ch_name, fontsize=8, pad=2)
    ax.grid(True, alpha=0.3)
    if ch_idx == 0:  # Add legend only to first subplot
        ax.legend(fontsize=6, loc='best')
    ax.set_xlabel('Time (s)', fontsize=8) if ch_idx >= (n_rows - 1) * n_cols else None
    ax.set_ylabel('% Change', fontsize=8) if ch_idx % n_cols == 0 else None

# Hide unused subplots
for idx in range(n_channels, len(axes)):
    axes[idx].axis('off')

plt.suptitle(f'HFA ERP for First {n_channels} Channels (Split by Response Type, Baselined to [-0.5, 0] s)', 
             fontsize=14, y=0.995)
plt.tight_layout()
plt.show()

# Summary plot showing mean HFA across first 196 channels, split by response
fig, ax = plt.subplots(1, 1, figsize=(10, 6))
for resp in unique_responses:
    if resp in hfa_erp_by_response:
        hfa_erp_resp = hfa_erp_by_response[resp]
        # Only average across first 196 channels
        mean_hfa_resp = np.mean(hfa_erp_resp[:n_channels, :], axis=0)  # average across channels
        std_hfa_resp = np.std(hfa_erp_resp[:n_channels, :], axis=0)
        sem_hfa_resp = std_hfa_resp / np.sqrt(n_channels)
        resp_mask = responses == resp
        
        ax.plot(times, mean_hfa_resp, linewidth=2, color=response_colors[resp], 
               label=f'Response: {resp} (n={resp_mask.sum()})')
        ax.fill_between(times, 
                       mean_hfa_resp - sem_hfa_resp,
                       mean_hfa_resp + sem_hfa_resp,
                       alpha=0.2, color=response_colors[resp])

ax.axhline(y=0, color='k', linestyle='--', linewidth=1, alpha=0.5)
ax.axvline(x=0, color='r', linestyle='--', linewidth=1, alpha=0.5, label='Image onset')
ax.axvspan(-0.5, 0, alpha=0.1, color='gray', label='Baseline period')
ax.set_xlabel('Time (s)', fontsize=12)
ax.set_ylabel('HFA (% change from baseline)', fontsize=12)
ax.set_title(f'Mean HFA ERP Across First {n_channels} Channels (Split by Response Type)', fontsize=14)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
sns.despine()
plt.tight_layout()
plt.show()

print(f"\nPlotted HFA ERP for first {n_channels} channels, split by {n_responses} response types")
for resp in unique_responses:
    if resp in hfa_erp_by_response:
        mean_hfa_resp = np.mean(hfa_erp_by_response[resp][:n_channels, :], axis=0)
        resp_mask = responses == resp
        print(f"  Response '{resp}': n={resp_mask.sum()}, range {np.min(mean_hfa_resp):.2f} to {np.max(mean_hfa_resp):.2f} % change")


In [ ]:
slope

In [ ]:
signals.shape

In [ ]:
segment._events[3].load()

In [ ]:
segment.analogsignals[0].shape

In [ ]:
segment.events[35].shape

In [ ]:
segment.events[35]

In [ ]:
plt.plot(signals[0:2048, 0])

In [ ]:
# get syncs
ttl = segment.events[0]

In [ ]:
syncs = ttl.load()

In [ ]:
ch_names

In [ ]:
syncs

In [ ]:
# load behavioral data
learn_df = pd.read_csv(f'{load_dir}/learn_df.csv')
combined_df = pd.read_csv(f'{load_dir}/combined_df.csv')

# Filter just to the participants in this notebook
learn_df = learn_df[learn_df.participant.isin(subj_ids)]
combined_df = combined_df[combined_df.participant.isin(subj_ids)]


# I enter the day information in stupidly 
combined_df.condition = combined_df.condition.apply(lambda x: 'Day1' if x=='day_1' else 'Day2' if x=='day_2' else np.nan)
combined_df['recog_time'] = combined_df['recog_time'].apply(lambda x: float(x) if (~pd.isna(x)) & (x!='None') else x)
combined_df['baseline_start_mem'] = combined_df['baseline_start_mem'].apply(lambda x: float(x) if (~pd.isna(x)) & (x!='None') else x)

# # add RPE to the combined df dataframe: 

combined_df['rpe'] = np.nan
for ix, row in learn_df.iterrows():
    combined_df.loc[(combined_df.img_path==row.img_path) & (combined_df.participant==row.participant), 'rpe'] = row.rpe


## Figure 3: Hippocampal oscillations encode RPE, while hippocampal HFA tracks memory encoding 

Let's: 

1. load the TFRs

2. restrict them to alpha, beta, and HFA

3. average within band

4. split by RPE magnitude 

5. plot

In [ ]:
# Explicitly define a list of analysis events and the baseline event. Should correspond to the dict


evs = {'gamble_start':[-1.0, 0], 
       'feedback_start': [0, 1.5],
       'baseline_start': [0, 0.75]}

baseline_ev = 'baseline_start'
analysis_evs = ['feedback_start']

day = 'Day1'

for i in analysis_evs: 
    if i not in evs.keys(): 
        raise ValueError('This analysis event is not in the dictionary')
        
# Define parameters for TFR: 
freqs = np.logspace(*np.log10([2, 200]), num=30)
n_cycles = np.floor(np.logspace(*np.log10([3, 10]), num=30))

In [ ]:
frequency_masks = {'theta': freqs<10,
                   'alpha': (freqs>=10) & (freqs<14),
                   'beta': (freqs>=14) & (freqs<30), 
                   'hfa': (freqs>=65)}
                   

In [ ]:
rois = ['HPC', 'OFC', 'dmPFC', 'ACC', 'AMY']

For plotting the power in time (not very effective): 

In [ ]:
# # Let's load the power epochs: 

# analysis_evs = ['feedback_start']

# # task condition to contrast 
# conditions = ['(rpe>0)',
#              '(rpe<0)']

# cond_name = 'rpe'

# tfr_group_data = []
# # {f'{a}': {f'{b}': [] for b in conditions} for a in rois}

# power_epochs = {f'{a}': {f'{b}': np.nan for b in analysis_evs} for a in subj_ids}

# for subj_id in subj_ids: 
#     for event in analysis_evs:
#         filepath = f'{base_dir}/projects/guLab/Salman/EphysAnalyses/{subj_id}/scratch/TFR'
#         power_epochs[subj_id][event] = mne.time_frequency.read_tfrs(f'{filepath}/{event}-tfr.h5')[0]
                
#         power_epochs[subj_id][event].metadata = learn_df[(learn_df.participant==subj_id)]
        
#         times = power_epochs[subj_id][event].times
            
#         # Get electrode df 
#         elec_df = pd.read_csv(f'{base_dir}/projects/guLab/Salman/EphysAnalyses/{subj_id}/Day1_reref_elec_df')
        
#         progress_bar = tqdm(elec_df.label, ascii=True, desc='Aggregating TFRs')

#         for elec_ix, chan in enumerate(progress_bar):
#             region = elec_df.salman_region.iloc[elec_ix]
#             if region in rois:
#                 for ix, cond in enumerate(conditions):
#                     for band in frequency_masks.keys():
#                         # Set the times for 
#                         times = power_epochs[subj_id][event].times
#                         plot_data = np.nanmean(np.nanmean(power_epochs[subj_id][event][cond].pick_channels([chan]).data, axis=0), axis=0)
#                         plot_data = np.nanmean(plot_data[frequency_masks[band], :], axis=0)
#                         plot_df = pd.DataFrame(columns=['times', 'power', 'band', 'condition', 'region'])
#                         plot_df['times'] = times
#                         plot_df['power'] = plot_data
#                         plot_df['band'] = band
#                         plot_df['condition'] = cond
#                         plot_df['region'] = region 
#                         plot_df['participant'] = subj_id
#                         tfr_group_data.append(plot_df)                    
                    

In [ ]:
# plot_dfs = pd.concat(tfr_group_data)

In [ ]:
# sns.lineplot(data=plot_dfs[plot_dfs.band=='beta'], x='times', y='power', hue='condition', errorbar='se')


In [ ]:
progress_bar = tqdm(subj_ids, ascii=True, desc='Saving TFR dataframes')
analysis_evs = ['feedback_start'] 

power_epochs = {f'{a}': {f'{b}': np.nan for b in analysis_evs} for a in subj_ids}


for subj_id in subj_ids:
    for day in ['Day1']:
        elec_df = pd.read_csv(f'{base_dir}/projects/guLab/Salman/EphysAnalyses/{subj_id}/{day}_reref_elec_df')
        for event in analysis_evs:

            filepath = f'{base_dir}/projects/guLab/Salman/EphysAnalyses/{subj_id}/scratch/TFR'
            if not os.path.exists(f'{filepath}/{event}/plots'):
                os.makedirs(f'{filepath}/{event}/plots')
            if not os.path.exists(f'{filepath}/{event}/dfs'):
                os.makedirs(f'{filepath}/{event}/dfs')

            power_epochs[subj_id][event] = mne.time_frequency.read_tfrs(f'{filepath}/{event}-tfr.h5')[0]

            # determine which epochs to analyze (all of them, in my case)
            # These epochs only grab all images from the gambling task: 

#             epochs_to_analyze = learn_df[(learn_df.participant==subj_id)].trials.values - 1

#             power_epochs[subj_id][event] = power_epochs[subj_id][event][epochs_to_analyze.astype(int)]

#             power_epochs[subj_id][event].metadata = learn_df[(learn_df.participant==subj_id)]

#             # These epochs only grab those images that were later involved in memory recognition: 
            epochs_to_analyze = combined_df[(combined_df.participant==subj_id) & (combined_df.condition=='Day1')].dropna(subset=['trials_gamble']).sort_values(by='trials_gamble').reset_index(drop=True).trials_gamble.values - 1

            power_epochs[subj_id][event] = power_epochs[subj_id][event][epochs_to_analyze.astype(int)]

            power_epochs[subj_id][event].metadata = combined_df[(combined_df.participant==subj_id) & (combined_df.condition=='Day1')].dropna(subset=['trials_gamble']).sort_values(by='trials_gamble').reset_index(drop=True)

            for chan in power_epochs[subj_id][event].ch_names:
                pow_df = power_epochs[subj_id][event].copy().pick_channels([chan]).to_data_frame()
                pow_df['fband'] = pow_df.freq.apply(lambda x: 'theta' if x<10 else 'alpha' if (x>=10) & (x<14) else 'beta' if (x>=14) & (x<30) else 'slowgamma' if (x>=30) & (x<55) else 'hfa')
                pow_df = pow_df.groupby(['fband','epoch']).mean().reset_index()
                
                tfr_df = pow_df.drop(columns=['time', 'freq']).rename(columns={'epoch':'trial', f'{chan}':'tfr'})

                power_epochs[subj_id][event].metadata['trial'] = tfr_df['trial'].unique()
                tfr_df = tfr_df.merge(power_epochs[subj_id][event].metadata, on=['trial'])

                tfr_df.insert(0,'channel', chan)
                tfr_df.insert(0,'region', elec_df[elec_df.label==chan].salman_region.values[0])
#                 tfr_df.insert(0,'participant', subj_id)
                tfr_df['event'] = event

                tfr_df.to_csv(f'{filepath}/{event}/dfs/{chan}_time_averaged_df_Day1.csv', index=False)


In [ ]:
chan_dfs = [] 
glms = []
elec_counter = 0 
for subj_id in tqdm(subj_ids):
    day = 'Day1'
    filepath = f'{base_dir}/projects/guLab/Salman/EphysAnalyses/{subj_id}/scratch/TFR'
    elec_df = pd.read_csv(f'{base_dir}/projects/guLab/Salman/EphysAnalyses/{subj_id}/{day}_reref_elec_df')
    for event in analysis_evs:
        for chan in elec_df.label.unique():
            # for chan in tqdm(ch_names, ascii=True, desc='Loading TFR dataframes'): 
            # chan_df = pd.read_csv(f'{filepath}/{event}/dfs/{chan}_time_averaged_df_SME.csv')
            chan_df = pd.read_csv(f'{filepath}/{event}/dfs/{chan}_time_averaged_df.csv')
                
            
            chan_df['unique_elec_id'] = elec_counter
            chan_df['zrpe'] = (chan_df.rpe - np.nanmean(chan_df.rpe)) / (2*np.nanstd(chan_df.rpe))
            chan_df['zpm'] = (chan_df.DPRIME - np.nanmean(chan_df.DPRIME)) / (2*np.nanstd(chan_df.DPRIME))
            try:
                chan_df['gamble_rt_log'] = np.log(chan_df['gamble_rt'])
            except KeyError:
                chan_df['gamble_rt_log'] = np.log(chan_df['gamble_rt_gamble'])
            chan_df['age'] = subj_df[subj_df.MSSMCode==subj_id].Age.values[0]
            chan_df['sex'] = subj_df[subj_df.MSSMCode==subj_id].Sex.str.strip().values[0]
            elec_counter +=1
            
            chan_dfs.append(chan_df)
        
chan_dfs = pd.concat(chan_dfs)

In [ ]:
plot_df = chan_dfs[chan_dfs.fband=='hfa']

First, let's plot the relationship between RPE and hippocampal power 

In [ ]:
# sns.lineplot(data=, x='times', y='power', hue='condition', errorbar='se')
f, ax = plt.subplots(1, 4, dpi=300, figsize=(12,3), sharex=True, sharey=True)


sns.regplot(data=plot_df[(plot_df.region=='HPC') & (plot_df.hits==True)].reset_index(drop=True),
            color = custom_palette[0], y_partial='zpm', truncate=True, units='participant',
            x='zrpe', y='tfr', ci=None, scatter=False, ax=ax[0])
sns.regplot(data=plot_df[(plot_df.region=='HPC') & (plot_df.hits==False)].reset_index(drop=True),
            color = custom_palette[1], y_partial='zpm', truncate=True, units='participant',
            x='zrpe', y='tfr', ci=None, scatter=False, ax=ax[0])

sns.regplot(data=plot_df[(plot_df.region=='dmPFC') & (plot_df.hits==True)].reset_index(drop=True),
            color = custom_palette[0], y_partial='zpm', truncate=True, units='participant',
            x='zrpe', y='tfr', ci=None, scatter=False, ax=ax[1])
sns.regplot(data=plot_df[(plot_df.region=='dmPFC') & (plot_df.hits==False)].reset_index(drop=True),
            color = custom_palette[1], y_partial='zpm', truncate=True, units='participant',
            x='zrpe', y='tfr', ci=None, scatter=False, ax=ax[1])

sns.regplot(data=plot_df[(plot_df.region=='OFC') & (plot_df.hits==True)].reset_index(drop=True), 
            color = custom_palette[0], y_partial='zpm', truncate=True, units='participant',
            x='zrpe', y='tfr', ci=None, scatter=False, ax=ax[2])
sns.regplot(data=plot_df[(plot_df.region=='OFC') & (plot_df.hits==False)].reset_index(drop=True), 
            color = custom_palette[1], y_partial='zpm', truncate=True, units='participant',
            x='zrpe', y='tfr', ci=None, scatter=False, ax=ax[2])


sns.regplot(data=plot_df[(plot_df.region=='AMY') & (plot_df.hits==True)].reset_index(drop=True), 
            color = custom_palette[0], y_partial='zpm', truncate=True, units='participant',
            x='zrpe', y='tfr', ci=None, scatter=False, ax=ax[3])
sns.regplot(data=plot_df[(plot_df.region=='AMY') & (plot_df.hits==False)].reset_index(drop=True), 
            color = custom_palette[1], y_partial='zpm', truncate=True, units='participant',
            x='zrpe', y='tfr', ci=None, scatter=False, ax=ax[3])

sns.despine()
f.tight_layout()

In [ ]:
# sns.lineplot(data=, x='times', y='power', hue='condition', errorbar='se')
f, ax = plt.subplots(1, 4, dpi=300, figsize=(12,3), sharex=True, sharey=True)


sns.regplot(data=plot_df[(plot_df.region=='HPC') & (plot_df.hits==True)].reset_index(drop=True),
            color = custom_palette[0], y_partial='zrpe', truncate=True, units='participant',
            x='zpm', y='tfr', ci=None, scatter=False, ax=ax[0])
sns.regplot(data=plot_df[(plot_df.region=='HPC') & (plot_df.hits==False)].reset_index(drop=True),
            color = custom_palette[1], y_partial='zrpe', truncate=True, units='participant',
            x='zpm', y='tfr', ci=None, scatter=False, ax=ax[0])

sns.regplot(data=plot_df[(plot_df.region=='dmPFC') & (plot_df.hits==True)].reset_index(drop=True),
            color = custom_palette[0], y_partial='zrpe', truncate=True, units='participant',
            x='zpm', y='tfr', ci=None, scatter=False, ax=ax[1])
sns.regplot(data=plot_df[(plot_df.region=='dmPFC') & (plot_df.hits==False)].reset_index(drop=True),
            color = custom_palette[1], y_partial='zrpe', truncate=True, units='participant',
            x='zpm', y='tfr', ci=None, scatter=False, ax=ax[1])

sns.regplot(data=plot_df[(plot_df.region=='OFC') & (plot_df.hits==True)].reset_index(drop=True), 
            color = custom_palette[0], y_partial='zrpe', truncate=True, units='participant',
            x='zpm', y='tfr', ci=None, scatter=False, ax=ax[2])
sns.regplot(data=plot_df[(plot_df.region=='OFC') & (plot_df.hits==False)].reset_index(drop=True), 
            color = custom_palette[1], y_partial='zrpe', truncate=True, units='participant',
            x='zpm', y='tfr', ci=None, scatter=False, ax=ax[2])


sns.regplot(data=plot_df[(plot_df.region=='AMY') & (plot_df.hits==True)].reset_index(drop=True), 
            color = custom_palette[0], y_partial='zrpe', truncate=True, units='participant',
            x='zpm', y='tfr', ci=None, scatter=False, ax=ax[3])
sns.regplot(data=plot_df[(plot_df.region=='AMY') & (plot_df.hits==False)].reset_index(drop=True), 
            color = custom_palette[1], y_partial='zrpe', truncate=True, units='participant',
            x='zpm', y='tfr', ci=None, scatter=False, ax=ax[3])

sns.despine()
f.tight_layout()

Plot the PSDs for every region 

In [ ]:
rois = ['OFC', 'dmPFC', 'HPC', 'AMY', 'ACC']

In [ ]:
# Let's load the power epochs for all : 

evs = {'gamble_start':[-1.0, 0], 
       'feedback_start': [0, 1.5],
       'baseline_start': [0, 0.75]}

baseline_ev = 'baseline_start'
analysis_evs = ['feedback_start']

day = 'Day1'

# Define parameters for TFR: 
freqs = np.logspace(*np.log10([2, 200]), num=30)
n_cycles = np.floor(np.logspace(*np.log10([3, 10]), num=30))

tfr_group_data = {f'{a}': [] for a in rois}

power_epochs = {f'{a}': {f'{b}': np.nan for b in analysis_evs} for a in subj_ids}

for subj_id in subj_ids: 
    for event in analysis_evs:
        filepath = f'{base_dir}/projects/guLab/Salman/EphysAnalyses/{subj_id}/scratch/TFR'
        
        save_path = f'{base_dir}/projects/guLab/Salman/EphysAnalyses/{subj_id}/neural/Day1'
        event_epochs_reref = mne.read_epochs(f'{save_path}/{event}-epo.fif', preload=True)
        ch_names = event_epochs_reref.ch_names
        #epochs_all_subjs_all_evs[subj_id][event]
            
        # Let's make sure we only do this for good channels
        good_chans = [x for x in event_epochs_reref.ch_names if x not in event_epochs_reref.info['bads']]
        picks = [x for x in good_chans]
        
        pow_struct = np.nan * np.ones([event_epochs_reref._data.shape[0], 
                               event_epochs_reref._data.shape[1], len(freqs), 
                               event_epochs_reref._data.shape[-1]])
    
        progress_bar = tqdm(np.arange(event_epochs_reref._data.shape[1]), ascii=True, desc='Computing TFR')

        for ch_ix in progress_bar: 
            ch_data = event_epochs_reref._data[:, ch_ix:ch_ix+1, :]
            bad_epochs  = np.where(event_epochs_reref.metadata[event_epochs_reref.ch_names[ch_ix]].notnull())[0]
            good_epochs = np.delete(np.arange(ch_data.shape[0]), bad_epochs)
            ch_data = np.delete(ch_data, bad_epochs, axis=0)
            ch_pow = mne.time_frequency.tfr_array_morlet(ch_data, sfreq=event_epochs_reref.info['sfreq'], 
                                                freqs=freqs, n_cycles=n_cycles, zero_mean=False, 
                                                use_fft=True, output='power', n_jobs=-1)

            pow_struct[good_epochs, ch_ix, :, :] = ch_pow[:, 0, :, :]
        
        temp_pow = mne.time_frequency.EpochsTFR(event_epochs_reref.info, pow_struct, 
                                                event_epochs_reref.times, freqs)
        
        temp_pow.crop(tmin=evs[event][0], tmax=evs[event][1])
        
        # Get electrode df 
        elec_df = pd.read_csv(f'{base_dir}/projects/guLab/Salman/EphysAnalyses/{subj_id}/Day1_reref_elec_df')
        
        progress_bar = tqdm(elec_df.label, ascii=True, desc='Aggregating TFRs')

        for elec_ix, chan in enumerate(progress_bar):
            region = elec_df.salman_region.iloc[elec_ix]
            if region in rois:
                # average across channels, events, and times
                plot_data = np.nanmean(np.nanmean(np.nanmean(temp_pow.copy().pick([chan]).data, axis=0), axis=0), axis=-1)
                tfr_group_data[region].append(plot_data)                    
                    

In [ ]:
def normalize_list(input_list):
    min_val = np.nanmin(input_list)
    max_val = np.nanmax(input_list)
    den = (max_val - min_val)
    if den == 0: 
        den = 0.001
    
    normalized_list = [(x - min_val) / den  for x in input_list]
    
    return normalized_list

In [ ]:
freqs = np.logspace(*np.log10([2, 200]), num=30)
save_dir_plots = '/Users/salmanqasim/Documents/Projects/MemBanditsEMU/Plots/MemoryBandits2_UAB_test'
os.makedirs(save_dir_plots, exist_ok=True)

for region in rois: 

    fig, ax = plt.subplots(1, 1, figsize=(4, 4), dpi=300)
    # Create a Pandas DataFrame
    plot_data = np.vstack(tfr_group_data[region])
    normalized_array = np.array([normalize_list(lst) for lst in plot_data])
    for ch in normalized_array:
        # plt.plot(freqs[:-1], np.log10(ch)[:-1], alpha=0.1, color='k')
        plot_power_spectra(freqs[:-1], ch[:-1], alpha =0.1,  
                       ax=ax, colors='black')   
    plot_power_spectra(freqs[:-1], np.nanmean(normalized_array[:, :-1], axis=0), 
                       ax=ax, colors='black')   
    # plt.plot(freqs[:-1], np.log10(np.nanmean(normalized_array[:, :-1], axis=0)), alpha=1, color='k')

    
    ax.set(title=f'{region}', xlabel='log frequency (hz)', 
           ylabel='log power (norm)', 
           xticks=[2, 10, 30, 50, 200], 
           xticklabels=[2, 10, 30, 50, 200])
    
    sns.despine()
    
    plt.savefig(f'{save_dir_plots}/feedback_PSD_{region}.pdf', dpi=300)

        
        

In [ ]:
# Let's load the power epochs for all : 

evs = {'recog_time':[-1.0, 0]}

# baseline_ev = 'baseline_start'
analysis_evs = ['recog_time']

day = 'Day1'

# Define parameters for TFR: 
freqs = np.logspace(*np.log10([2, 200]), num=30)
n_cycles = np.floor(np.logspace(*np.log10([3, 10]), num=30))

tfr_group_data = {f'{a}': [] for a in rois}

power_epochs = {f'{a}': {f'{b}': np.nan for b in analysis_evs} for a in subj_ids}

for subj_id in subj_ids: 
    if subj_id in ['MS024', 'MS034', 'MS038']:
        continue
    for event in analysis_evs:
        filepath = f'{base_dir}/projects/guLab/Salman/EphysAnalyses/{subj_id}/scratch/TFR'
        
        save_path = f'{base_dir}/projects/guLab/Salman/EphysAnalyses/{subj_id}/neural/Day1'
        event_epochs_reref = mne.read_epochs(f'{save_path}/{event}-epo.fif', preload=True)
        ch_names = event_epochs_reref.ch_names
        #epochs_all_subjs_all_evs[subj_id][event]
            
        # Let's make sure we only do this for good channels
        good_chans = [x for x in event_epochs_reref.ch_names if x not in event_epochs_reref.info['bads']]
        picks = [x for x in good_chans]
        
        pow_struct = np.nan * np.ones([event_epochs_reref._data.shape[0], 
                               event_epochs_reref._data.shape[1], len(freqs), 
                               event_epochs_reref._data.shape[-1]])
    
        progress_bar = tqdm(np.arange(event_epochs_reref._data.shape[1]), ascii=True, desc='Computing TFR')

        for ch_ix in progress_bar: 
            ch_data = event_epochs_reref._data[:, ch_ix:ch_ix+1, :]
            bad_epochs  = np.where(event_epochs_reref.metadata[event_epochs_reref.ch_names[ch_ix]].notnull())[0]
            good_epochs = np.delete(np.arange(ch_data.shape[0]), bad_epochs)
            ch_data = np.delete(ch_data, bad_epochs, axis=0)
            ch_pow = mne.time_frequency.tfr_array_morlet(ch_data, sfreq=event_epochs_reref.info['sfreq'], 
                                                freqs=freqs, n_cycles=n_cycles, zero_mean=False, 
                                                use_fft=True, output='power', n_jobs=-1)

            pow_struct[good_epochs, ch_ix, :, :] = ch_pow[:, 0, :, :]
        
        temp_pow = mne.time_frequency.EpochsTFR(event_epochs_reref.info, pow_struct, 
                                                event_epochs_reref.times, freqs)
        
        temp_pow.crop(tmin=evs[event][0], tmax=evs[event][1])
        
        # Get electrode df 
        elec_df = pd.read_csv(f'{base_dir}/projects/guLab/Salman/EphysAnalyses/{subj_id}/Day1_reref_elec_df')
        
        progress_bar = tqdm(elec_df.label, ascii=True, desc='Aggregating TFRs')

        for elec_ix, chan in enumerate(progress_bar):
            region = elec_df.salman_region.iloc[elec_ix]
            if region in rois:
                # average across channels, events, and times
                plot_data = np.nanmean(np.nanmean(np.nanmean(temp_pow.copy().pick_channels([chan]).data, axis=0), axis=0), axis=-1)
                tfr_group_data[region].append(plot_data)                    
                    

In [ ]:
freqs = np.logspace(*np.log10([2, 200]), num=30)
save_dir_plots = '/Users/salmanqasim/Documents/Projects/MemBanditsEMU/Plots/MemoryBandits2_UAB_test'
os.makedirs(save_dir_plots, exist_ok=True)

for region in rois: 

    fig, ax = plt.subplots(1, 1, figsize=(4, 4), dpi=300)
    # Create a Pandas DataFrame
    plot_data = np.vstack(tfr_group_data[region])
    normalized_array = np.array([normalize_list(lst) for lst in plot_data])
    for ch in normalized_array:
        # plt.plot(freqs[:-1], np.log10(ch)[:-1], alpha=0.1, color='k')
        plot_power_spectra(freqs[:-1], ch[:-1], alpha =0.1,  
                       ax=ax, colors='black')   
    plot_power_spectra(freqs[:-1], np.nanmean(normalized_array[:, :-1], axis=0), 
                       ax=ax, colors='black')   
    # plt.plot(freqs[:-1], np.log10(np.nanmean(normalized_array[:, :-1], axis=0)), alpha=1, color='k')

    
    ax.set(title=f'{region}', xlabel='log frequency (hz)', 
           ylabel='log power (norm)', 
           xticks=[2, 10, 30, 50, 200], 
           xticklabels=[2, 10, 30, 50, 200])
    
    sns.despine()
    
    plt.savefig(f'{save_dir_plots}/recog_PSD_{region}.pdf', dpi=300)

        
        

 I have to plot the bayesian regression results in another noteboko that loads the proper environment for that type of analysis : BayesianRegression.ipynb